
# P2_84 — Complete T4 Answer-Code Permutation Sensitivity

**Purpose.** Complete the Qwen3-8B T4 sensitivity analysis by enumerating **all 6 bijective mappings** between abstract answer codes `A/B/C` and severity labels `1/2/3` under both frozen rubric phrasings `P1` and `P2`.

The previously executed set covered `123` (identity), `231`, and `312`. This notebook runs the **three missing transpositions** `132`, `213`, and `321`, reuses any existing outputs, and then recomputes the full 12-variant T4 summary.

This is a **post-hoc validity/sensitivity analysis**, not a search for the best prompt and not a new primary endpoint.


In [ ]:

%pip -q install -U "transformers>=4.57,<5" accelerate bitsandbytes safetensors pandas openpyxl "scikit-learn>=1.4"


In [ ]:

from pathlib import Path
import os, sys, json, hashlib, zipfile, shutil
import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

PROJECT_DIR = Path('/content/drive/MyDrive/P2') if IN_COLAB else Path('/content/P2')
INPUT_DIR = PROJECT_DIR/'input'
RESULTS_DIR = PROJECT_DIR/'results'
OUT_DIR = RESULTS_DIR/'P2_81_M4_PROTOCOL_SENSITIVITY'
LOCAL_CACHE = Path('/content/p284_m4_t4_fullperm_cache')
for d in [INPUT_DIR, RESULTS_DIR, OUT_DIR, LOCAL_CACHE]: d.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(LOCAL_CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(LOCAL_CACHE/'transformers')

GOLD_NAME='P2_FINAL_SENIOR_ADJUDICATED_GOLD_v1.0.xlsx'
BASE_NAME='P2_CPU_Baselines_v1.0.xlsx'
EXPECTED={
    GOLD_NAME:'ccc910b7bafbfa9607e93ef2eba8605f56f7ed87460ef74892d4877c7db4de65',
    BASE_NAME:'4502e71bd6d1b6d2941b0b10650caaa433d7188bf958017d2fc646d328a31163',
}
MODEL_ID='Qwen/Qwen3-8B'
MODEL_REVISION='e8bbd8252970581ea5b08b6a5b3e668adaf3161a'
BATCH_SIZE=4
BOOT_REPS=2000
SEED=20260908

def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for ch in iter(lambda:f.read(1024*1024),b''): h.update(ch)
    return h.hexdigest()

for n,e in EXPECTED.items():
    p=INPUT_DIR/n
    assert p.exists(), f'Missing {p}'
    o=sha256(p); print(n,o); assert o==e, 'Frozen input hash mismatch.'

gold=pd.read_excel(INPUT_DIR/GOLD_NAME,sheet_name='FINAL_GOLD')
gold_by_case=gold.set_index('Case_ID',drop=False)
oof_t4=pd.read_excel(INPUT_DIR/BASE_NAME,sheet_name='OOF_T4')
assert len(oof_t4)==474

assert torch.cuda.is_available(), 'Select a GPU runtime.'
print('GPU:',torch.cuda.get_device_name(0),'transformers:',transformers.__version__)



## Optional recovery of the previous P2_81 sensitivity outputs

If the original folder `results/P2_81_M4_PROTOCOL_SENSITIVITY` is still in Drive, nothing is needed. If not, upload the previously returned `P2_81_M4_PROTOCOL_SENSITIVITY_RESULTS.zip`; the cell below can extract it into `OUT_DIR`.


In [ ]:

# Optional: only run the upload branch if the old T4 identity file is missing.
required_old = OUT_DIR/'OOF_M4_SENS_T4_P1_identity.csv'
if not required_old.exists() and IN_COLAB:
    from google.colab import files
    print('Previous T4 outputs not found. Upload P2_81_M4_PROTOCOL_SENSITIVITY_RESULTS.zip')
    up=files.upload()
    zname=next((n for n in up if n.endswith('.zip')),None)
    assert zname is not None, 'No ZIP uploaded.'
    zpath=Path('/content')/zname
    zpath.write_bytes(up[zname])
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(OUT_DIR)
print('Existing old reference:', required_old.exists())


In [ ]:

# Load frozen model and single-token codes.
dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=dtype)
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REVISION)
tokenizer.padding_side='left'
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REVISION,quantization_config=quant,device_map='auto')
model.eval()
CODE_ID={}
for c in ['A','B','C']:
    ids=tokenizer(c,add_special_tokens=False)['input_ids']
    print(c,ids); assert len(ids)==1
    CODE_ID[c]=ids[0]


In [ ]:
def serialise_C(g):
    p=str(g['Prompt_Text'] if pd.notna(g['Prompt_Text']) else '').strip()
    s=str(g['Source_Context'] if pd.notna(g['Source_Context']) else '').strip()
    o=str(g['Model_Output'] if pd.notna(g['Model_Output']) else '').strip()
    if str(g['Source_Available']).lower()=='yes' and s:
        return f"Task:\n{p}\n\nAuthorised source/context:\n{s}\n\nResponse:\n{o}"
    return f"Task:\n{p}\n\nResponse:\n{o}"

MEANINGS={
1:'severity 1 (Minor): localized disruption with limited effect on the main answer.',
2:'severity 2 (Moderate): materially weakens an important task-relevant relation or component.',
3:'severity 3 (Severe): undermines the main answer, reverses meaning, contradicts the prompt/source, or makes the answer unreliable for the task.',
}
SYSTEM_P1=("You are a conservative semantic-coherence evaluator.\n"
           "Apply only the supplied target rubric to the visible text.\n"
           "Do not infer hidden facts. Do not output an explanation.\n"
           "Your decision space is closed: select exactly one of the permitted answer codes.")
SYSTEM_P2=("Classify the visible case using only the criterion stated below.\n"
           "Use no outside knowledge and do not infer evidence that is not visible.\n"
           "Return a single permitted code only; do not provide reasoning or any additional text.")
P1_INTRO='The case is already known to contain a failure. Judge functional severity, not confidence.'
P1_END='Choose only A, B, or C.'

def rubric(mapping,template):
    lines=[f'{code} = {MEANINGS[label]}' for code,label in mapping.items()]
    if template=='P1':
        return P1_INTRO+'\n'+'\n'.join(lines)+'\n'+P1_END
    intro='A failure is already present. Rate its functional impact on the requested task rather than your confidence.'
    return ('Criterion: '+intro+'\n\nCode definitions:\n'+'\n'.join(lines)+
            '\n\nSelect the one code whose definition best matches the visible case.')

def make_chat(g,mapping,template):
    sysmsg=SYSTEM_P1 if template=='P1' else SYSTEM_P2
    user=rubric(mapping,template)+'\n\nVisible case:\n'+serialise_C(g)
    return tokenizer.apply_chat_template(
        [{'role':'system','content':sysmsg},{'role':'user','content':user}],
        tokenize=False,add_generation_prompt=True,enable_thinking=False)

@torch.inference_mode()
def score_batch(texts,codes):
    enc=tokenizer(texts,return_tensors='pt',padding=True,truncation=True,max_length=4096)
    enc={k:v.to(model.device) for k,v in enc.items()}
    out=model(**enc)
    logp=torch.log_softmax(out.logits[:,-1,:].float(),dim=-1)
    ids=torch.tensor([CODE_ID[c] for c in codes],device=logp.device)
    return logp[:,ids].detach().cpu().numpy()



## All six T4 bijections

Mappings are written as the severity labels attached to `(A,B,C)`:

- `123` identity
- `132`, `213`, `321` = the three transpositions (missing in P2_81)
- `231`, `312` = the two three-cycles (already run in P2_81)


In [ ]:

PERMS={
    'identity': {'A':1,'B':2,'C':3},      # 123
    'perm_132': {'A':1,'B':3,'C':2},
    'perm_213': {'A':2,'B':1,'C':3},
    'cycle_231': {'A':2,'B':3,'C':1},
    'cycle_312': {'A':3,'B':1,'C':2},
    'perm_321': {'A':3,'B':2,'C':1},
}
MISSING=['perm_132','perm_213','perm_321']

def run_variant(template,perm_name,mapping):
    out_path=OUT_DIR/f'OOF_M4_SENS_T4_{template}_{perm_name}.csv'
    rows=oof_t4.copy(); codes=list(mapping.keys())
    existing={}
    if out_path.exists():
        old=pd.read_csv(out_path)
        for _,r in old.iterrows(): existing[str(r.Case_ID)]=r.to_dict()
        print('resume',template,perm_name,len(existing))
    pending=[r for _,r in rows.iterrows() if str(r.Case_ID) not in existing]
    records=list(existing.values())
    for start in range(0,len(pending),BATCH_SIZE):
        batch=pending[start:start+BATCH_SIZE]
        texts=[make_chat(gold_by_case.loc[str(r.Case_ID)],mapping,template) for r in batch]
        scores=score_batch(texts,codes)
        for r,sc in zip(batch,scores):
            j=int(np.argmax(sc)); code=codes[j]; pred=int(mapping[code]); ss=np.sort(sc)
            records.append({
                'Case_ID':str(r.Case_ID),'Prompt_ID':r.Prompt_ID,'Task_Type':r.Task_Type,
                'Fold':int(r.Fold),'y_true':int(r.y_true),'template':template,'permutation':perm_name,
                'M4_pred':pred,'M4_code':code,
                'M4_code_scores':json.dumps({c:float(v) for c,v in zip(codes,sc)}),
                'M4_score_margin':float(ss[-1]-ss[-2]),
            })
        pd.DataFrame(records).drop_duplicates('Case_ID',keep='last').to_csv(out_path,index=False)
        if start % 100 == 0:
            print(template,perm_name,min(start+BATCH_SIZE,len(pending)),'/',len(pending))
    d=pd.read_csv(out_path).drop_duplicates('Case_ID',keep='last')
    assert len(d)==474 and d.Case_ID.astype(str).nunique()==474
    return d

# Run only the three missing mappings under P1 and P2.
for template in ['P1','P2']:
    for pname in MISSING:
        print('RUN',template,pname)
        run_variant(template,pname,PERMS[pname])


In [ ]:

# Normalise legacy filenames, then load all 12 T4 variants.
legacy={
    'identity':'identity',
    'cycle_231':'cycle_231',
    'cycle_312':'cycle_312',
}
all_runs={}
for template in ['P1','P2']:
    for pname in PERMS:
        f=OUT_DIR/f'OOF_M4_SENS_T4_{template}_{pname}.csv'
        assert f.exists(), f'Missing {f}. If this is an old variant, extract the P2_81 ZIP first.'
        d=pd.read_csv(f).drop_duplicates('Case_ID',keep='last')
        assert len(d)==474
        all_runs[(template,pname)]=d
print('Loaded variants:',len(all_runs))


In [ ]:

# Compute full-permutation metrics, class distributions, agreement with reference, and clustered CIs.
def macro_from_cm(cm):
    tp=np.diag(cm).astype(float); fp=cm.sum(0)-tp; fn=cm.sum(1)-tp; den=2*tp+fp+fn
    return float(np.divide(2*tp,den,out=np.zeros_like(tp),where=den!=0).mean())
def cluster_cms(df,pred,labels):
    ix={v:i for i,v in enumerate(labels)}; arr=[]
    for _,s in df.assign(_pid=df.Prompt_ID.astype(str)).groupby('_pid',sort=True):
        cm=np.zeros((len(labels),len(labels)),int)
        for y,p in zip(s.y_true.astype(int),s[pred].astype(int)): cm[ix[y],ix[p]]+=1
        arr.append(cm)
    return np.stack(arr)
def ci(df,pred='M4_pred',labels=[1,2,3],reps=BOOT_REPS):
    cms=cluster_cms(df,pred,labels); rng=np.random.default_rng(SEED); vals=[]
    for _ in range(reps):
        q=rng.choice(len(cms),size=len(cms),replace=True)
        vals.append(macro_from_cm(cms[q].sum(0)))
    return np.percentile(vals,[2.5,97.5])

ref=all_runs[('P1','identity')][['Case_ID','M4_pred']].rename(columns={'M4_pred':'ref_pred'})
metric_rows=[]; dist_rows=[]; agree_rows=[]
for (template,pname),d in all_runs.items():
    lo,hi=ci(d)
    metric_rows.append({
        'target':'T4','template':template,'permutation':pname,'n':len(d),
        'macro_f1':f1_score(d.y_true,d.M4_pred,labels=[1,2,3],average='macro',zero_division=0),
        'accuracy':accuracy_score(d.y_true,d.M4_pred),
        'qwk':cohen_kappa_score(d.y_true,d.M4_pred,weights='quadratic'),
        'ci_low':lo,'ci_high':hi,
    })
    vc=d.M4_pred.astype(int).value_counts().reindex([1,2,3],fill_value=0)
    for lab,n in vc.items():
        dist_rows.append({'target':'T4','template':template,'permutation':pname,'predicted_label':lab,'n':int(n),'proportion':float(n/len(d))})
    a=d[['Case_ID','M4_pred']].merge(ref,on='Case_ID',validate='one_to_one')
    agree_rows.append({'target':'T4','template':template,'permutation':pname,'agreement_with_P1_identity':float((a.M4_pred==a.ref_pred).mean())})

metrics=pd.DataFrame(metric_rows).sort_values(['template','permutation'])
dists=pd.DataFrame(dist_rows).sort_values(['template','permutation','predicted_label'])
agreement=pd.DataFrame(agree_rows).sort_values(['template','permutation'])

summary=pd.DataFrame([{
    'target':'T4','variants':len(metrics),
    'macro_f1_mean':metrics.macro_f1.mean(),'macro_f1_sd':metrics.macro_f1.std(ddof=1),
    'macro_f1_min':metrics.macro_f1.min(),'macro_f1_max':metrics.macro_f1.max(),
    'qwk_mean':metrics.qwk.mean(),'qwk_min':metrics.qwk.min(),'qwk_max':metrics.qwk.max(),
    'P1_identity_macro_f1':float(metrics.query("template=='P1' and permutation=='identity'").macro_f1.iloc[0]),
}])

metrics.to_csv(OUT_DIR/'P2_84_T4_ALL_PERMUTATION_METRICS.csv',index=False)
dists.to_csv(OUT_DIR/'P2_84_T4_ALL_PERMUTATION_CLASS_DISTRIBUTIONS.csv',index=False)
agreement.to_csv(OUT_DIR/'P2_84_T4_ALL_PERMUTATION_AGREEMENT.csv',index=False)
summary.to_csv(OUT_DIR/'P2_84_T4_ALL_PERMUTATION_SUMMARY.csv',index=False)

display(metrics)
display(summary)


In [ ]:

# Package only the new complete-T4 outputs plus all six-by-two OOF files.
zip_path=RESULTS_DIR/'P2_84_M4_T4_COMPLETE_PERMUTATION_RESULTS.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for template in ['P1','P2']:
        for pname in PERMS:
            f=OUT_DIR/f'OOF_M4_SENS_T4_{template}_{pname}.csv'
            z.write(f,arcname=f.name)
    for name in [
        'P2_84_T4_ALL_PERMUTATION_METRICS.csv',
        'P2_84_T4_ALL_PERMUTATION_CLASS_DISTRIBUTIONS.csv',
        'P2_84_T4_ALL_PERMUTATION_AGREEMENT.csv',
        'P2_84_T4_ALL_PERMUTATION_SUMMARY.csv',
    ]:
        z.write(OUT_DIR/name,arcname=name)
print('Created:',zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
